In [7]:
import sys
sys.path.append("../src")

In [8]:
from utils.connections import get_target_db

target_db = get_target_db()
target_db.connect()

Connected!


1. Write SQL to get total sales per month by region. 


In [9]:
QUERY = """
select
	ds.location,
	dd.month,
	COUNT(DISTINCT fs.sale_id) as total_sales
from
	dim_store ds
inner join fact_sales fs on
	ds.store_key = fs.store_key
inner join dim_date dd on
	dd.date_key = fs.date_key
group by
	ds.location,
	dd.month
order by
    ds.location,
	dd.month;
"""

sales_by_region = target_db.query_df(QUERY)
sales_by_region

,location,month,total_sales
0,Adamstad,1,91
1,Adamstad,2,69
2,Adamstad,3,97
3,Adamstad,4,72
4,Adamstad,5,78
...,...,...,...
1195,Williamsport,8,81
1196,Williamsport,9,81
1197,Williamsport,10,87
1198,Williamsport,11,84


2. Top 5 selling products by revenue.

In [4]:
QUERY = """
with product_revenue as (
select
	dp.product_name,
	SUM(fs.total_amount) as total_revenue
from
	dim_product dp
inner join fact_sales fs on
	dp.product_key = fs.product_key
group by
	dp.product_name
)
select
	product_name,
	total_revenue,
	rank() over (
	order by total_revenue desc) as revenue_rank
from
	product_revenue;
"""

top_products = target_db.query_df(QUERY)
top_products

,product_name,total_revenue,revenue_rank
0,Be,1592758.98,1
1,Benefit,1565813.00,2
2,White,1557053.72,3
3,Six,1542616.36,4
4,Moment,1510923.52,5
...,...,...,...
389,Today,421515.96,390
390,Admit,419394.79,391
391,Wear,410133.49,392
392,Store,396275.49,393


3. Calculate Customer Retention

In [6]:
QUERY = """
WITH monthly_customers AS (
    SELECT
        d.year,
        d.month,
        fs.customer_key
    FROM fact_sales fs
    JOIN dim_date d ON fs.date_key = d.date_key
    GROUP BY d.year, d.month, fs.customer_key
),

retention AS (
    SELECT
        curr.year AS current_year,
        curr.month AS current_month,
        COUNT(DISTINCT curr.customer_key) AS customers_in_month,
        COUNT(DISTINCT next.customer_key) AS retained_next_month
    FROM monthly_customers curr
    LEFT JOIN monthly_customers next
        ON curr.customer_key = next.customer_key
        AND (
            (curr.year = next.year AND curr.month + 1 = next.month) OR
            (curr.year + 1 = next.year AND curr.month = 12 AND next.month = 1)
        )
    GROUP BY curr.year, curr.month
)

SELECT
    current_year,
    current_month,
    customers_in_month,
    retained_next_month,
    ROUND(100.0 * retained_next_month / customers_in_month, 2) AS retention_rate_percent
FROM retention
ORDER BY current_year, current_month;
"""


customer_retention = target_db.query_df(QUERY)
customer_retention

,current_year,current_month,customers_in_month,retained_next_month,retention_rate_percent
0,2020,1,1310,185,14.12
1,2020,2,1270,140,11.02
2,2020,3,1296,156,12.04
3,2020,4,1316,186,14.13
4,2020,5,1347,175,12.99
...,...,...,...,...,...
67,2025,8,1329,175,13.17
68,2025,9,1276,163,12.77
69,2025,10,1298,163,12.56
70,2025,11,1242,152,12.24
